# 02 - Transformación y limpieza de datos
## Contratos Menores - Ayuntamiento de Vilagarcía de Arousa

Este notebook toma el fichero crudo descargado en el paso anterior y produce un dataset limpio, tipado y listo para cargar en Snowflake.

**Entrada:** `data/raw/contratos_menores.xlsx`  
**Salida:** `data/processed/contratos_limpios.csv`

---

### Decisiones y supuestos documentados

| # | Decisión | Justificación |
|---|---|---|
| 1 | Se eliminan `Unnamed: 0` y `Lista de lotes` | 100% y 99.8% nulas respectivamente. Sin valor analítico. |
| 2 | `Fecha formalización` (96% nula) se sustituye por año extraído del número de referencia | El número de referencia sigue el patrón `CT NNN/AAAA`, por lo que el año es recuperable. |
| 3 | `Código CPV` (96% nulo) se conserva tal cual | Podría ser útil para análisis futuros; se anota el alto % de nulos. |
| 4 | Se separa `Contratistas` en `nif_contratista` y `nombre_contratista` | El campo mezcla CIF/NIF con nombre en un solo string (`"B27188317 - EXCAVACIONES J.CARREIRA"`). |
| 5 | Los importes (strings con formato español `"1.234,56"`) se convierten a `float` | Requiere quitar los puntos de miles y sustituir la coma decimal por punto. |
| 6 | Se conservan los `Contrato Mayor` junto a los `Contrato Menor` | El dataset oficial los incluye a todos; se añade una columna booleana `es_contrato_menor` para filtrar fácilmente. |
| 7 | Los nombres de columna se normalizan a snake_case sin tildes ni espacios | Facilita el trabajo en SQL y Python. |

## 0 — Importaciones y rutas

In [ ]:
import re
import pandas as pd
from pathlib import Path

RUTA_RAW       = Path("../data/raw/contratos_menores.xlsx")
RUTA_PROCESADO = Path("../data/processed/contratos_limpios.csv")

print(f"Leyendo: {RUTA_RAW}")

## 1 — Cargar el fichero crudo

In [ ]:
df = pd.read_excel(RUTA_RAW, dtype=str)  # dtype=str para no perder ceros a la izquierda en CIFs

print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
df.head(3)

## 2 — Eliminar columnas sin valor

In [ ]:
COLUMNAS_BASURA = ["Unnamed: 0", "Lista de lotes"]
df = df.drop(columns=COLUMNAS_BASURA)

print(f"Columnas tras limpieza: {df.shape[1]}")
print(df.columns.tolist())

## 3 — Renombrar columnas a snake_case

In [ ]:
RENOMBRAR = {
    "Entidad Contratante":                               "entidad_contratante",
    "Número de Referencia del Contrato":                  "num_referencia",
    "Objeto del Contrato":                               "objeto_contrato",
    "Valor estimado del contrato (en euros)":             "valor_estimado_eur",
    "Presupuesto base sin impuestos (en euros)":          "presupuesto_base_eur",
    "Plazo ejecución (en meses)":                        "plazo_meses",
    "Código CPV del objeto del contrato":                 "codigo_cpv",
    "Tipo de contratación":                              "tipo_contratacion",
    "Tipo de Contrato":                                  "tipo_contrato",
    "Tipo de procedimiento":                             "tipo_procedimiento",
    "Sistema de contratación":                           "sistema_contratacion",
    "Tramitación":                                       "tramitacion",
    "Fecha formalización":                               "fecha_formalizacion",
    "Importe total ofertado (sin impuestos) (en euros)": "importe_sin_iva_eur",
    "Importe total ofertado (con impuestos) (en euros)": "importe_con_iva_eur",
    "URL a la licitación específica del expediente":      "url_licitacion",
    "Observaciones":                                     "observaciones",
    "Contratistas":                                      "contratistas_raw",
}

df = df.rename(columns=RENOMBRAR)
print(df.columns.tolist())

## 4 — Limpiar e importar la columna de contratistas

Formato original: `"B27188317 - EXCAVACIONES J.CARREIRA, S.L."`  
Se separa por el primer ` - ` en NIF/CIF y nombre.

In [ ]:
def separar_contratista(texto):
    """Devuelve (nif, nombre) a partir de 'NIF - NOMBRE'."""
    if pd.isna(texto) or not isinstance(texto, str):
        return pd.NA, pd.NA
    partes = texto.split(" - ", maxsplit=1)
    if len(partes) == 2:
        return partes[0].strip(), partes[1].strip()
    return pd.NA, texto.strip()

df[["nif_contratista", "nombre_contratista"]] = (
    df["contratistas_raw"]
    .apply(lambda x: pd.Series(separar_contratista(x)))
)

# Verificación
print("Ejemplos de separación:")
df[["contratistas_raw", "nif_contratista", "nombre_contratista"]].head(5)

## 5 — Convertir importes de string español a float

Formato español: `"1.234,56"` → `1234.56`  
Proceso: quitar puntos de miles, sustituir coma decimal por punto, convertir a float.

In [ ]:
def euros_a_float(valor):
    """Convierte un string de importe en formato español a float."""
    if pd.isna(valor) or str(valor).strip() in ("", "nan"):
        return pd.NA
    limpio = str(valor).strip().replace(".", "").replace(",", ".")
    try:
        return float(limpio)
    except ValueError:
        return pd.NA

COLS_IMPORTE = ["valor_estimado_eur", "presupuesto_base_eur", "importe_sin_iva_eur", "importe_con_iva_eur"]

for col in COLS_IMPORTE:
    df[col] = df[col].apply(euros_a_float).astype("Float64")

print("Tipos resultantes:")
print(df[COLS_IMPORTE].dtypes)
print("\nEjemplos:")
df[COLS_IMPORTE].head()

## 6 — Extraer el año del número de referencia

`Fecha formalización` tiene un 96% de nulos, pero el **número de referencia** incluye siempre el año:  
`CT 80/2022-e` → **2022** | `CT 02/2023` → **2023**

Extraemos ese año como columna `anio_contrato`, que usaremos para análisis temporales.

In [ ]:
def extraer_anio(referencia):
    """Extrae el año de 4 dígitos de una referencia tipo 'CT 80/2022-e'."""
    if pd.isna(referencia):
        return pd.NA
    match = re.search(r"/(\d{4})", str(referencia))
    return int(match.group(1)) if match else pd.NA

df["anio_contrato"] = df["num_referencia"].apply(extraer_anio).astype("Int64")

print("Distribución por año:")
print(df["anio_contrato"].value_counts().sort_index())
print(f"\nReferencias sin año extraíble: {df['anio_contrato'].isna().sum()}")

## 7 — Normalizar `fecha_formalizacion`

Solo un 4% de registros tiene fecha. La convertimos a `datetime` cuando existe.

In [ ]:
df["fecha_formalizacion"] = pd.to_datetime(df["fecha_formalizacion"], dayfirst=True, errors="coerce")

registros_con_fecha = df["fecha_formalizacion"].notna().sum()
print(f"Registros con fecha_formalizacion: {registros_con_fecha} ({registros_con_fecha/len(df)*100:.1f}%)")
print("\nEjemplos de fechas parseadas:")
df.loc[df["fecha_formalizacion"].notna(), ["num_referencia", "fecha_formalizacion"]].head()

## 8 — Añadir columna `es_contrato_menor`

In [ ]:
df["es_contrato_menor"] = df["tipo_contratacion"].str.strip() == "Contrato Menor"

print("Distribución tipo_contratacion:")
print(df["tipo_contratacion"].value_counts())
print(f"\nContratos menores: {df['es_contrato_menor'].sum()}")
print(f"Contratos mayores: {(~df['es_contrato_menor']).sum()}")

## 9 — Convertir `plazo_meses` a numérico

In [ ]:
df["plazo_meses"] = pd.to_numeric(df["plazo_meses"], errors="coerce").astype("Float64")

print(df["plazo_meses"].describe())

## 10 — Revisar contratos cercanos al límite legal

Los contratos menores tienen límites legales:  
- Servicios y suministros: **≤ 15.000 €**  
- Obras: **≤ 40.000 €**

Marcamos los contratos que superan esos límites o que se acercan sospechosamente (≥ 90% del límite).

In [ ]:
LIMITE_SERVICIOS = 15_000
LIMITE_OBRAS     = 40_000
UMBRAL_ALERTA    = 0.90  # 90% del límite

def flag_limite(row):
    if not row["es_contrato_menor"]:
        return "no_aplica"
    importe = row["presupuesto_base_eur"]
    tipo    = str(row["tipo_contrato"]).strip()
    if pd.isna(importe):
        return "sin_importe"
    limite = LIMITE_OBRAS if tipo == "Obras" else LIMITE_SERVICIOS
    if importe > limite:
        return "supera_limite"
    if importe >= limite * UMBRAL_ALERTA:
        return "cerca_del_limite"
    return "ok"

df["flag_limite"] = df.apply(flag_limite, axis=1)

print("Distribución de flags de límite legal:")
print(df["flag_limite"].value_counts())

print("\nContratos que superan el límite legal:")
df.loc[df["flag_limite"] == "supera_limite",
       ["num_referencia", "objeto_contrato", "tipo_contrato", "presupuesto_base_eur", "nombre_contratista"]]

## 11 — Revisión final de calidad

In [ ]:
resumen = pd.DataFrame({
    "tipo":     df.dtypes,
    "nulos":    df.isnull().sum(),
    "% nulos":  (df.isnull().mean() * 100).round(1),
    "únicos":   df.nunique(),
})
print(resumen.to_string())

In [ ]:
# Estadísticas básicas de los importes
print("=== Estadísticas de importes (todos los contratos) ===")
print(df[COLS_IMPORTE].describe().round(2))

print("\n=== Solo contratos menores ===")
print(df.loc[df["es_contrato_menor"], COLS_IMPORTE].describe().round(2))

## 12 — Ordenar columnas y guardar

In [ ]:
ORDEN_COLUMNAS = [
    # Identificación
    "num_referencia",
    "anio_contrato",
    "fecha_formalizacion",
    "entidad_contratante",
    # Clasificación
    "tipo_contratacion",
    "es_contrato_menor",
    "tipo_contrato",
    "tipo_procedimiento",
    "sistema_contratacion",
    "tramitacion",
    # Descripción
    "objeto_contrato",
    "codigo_cpv",
    "plazo_meses",
    # Importes
    "valor_estimado_eur",
    "presupuesto_base_eur",
    "importe_sin_iva_eur",
    "importe_con_iva_eur",
    "flag_limite",
    # Contratista
    "nif_contratista",
    "nombre_contratista",
    "contratistas_raw",
    # Extra
    "observaciones",
    "url_licitacion",
]

df = df[ORDEN_COLUMNAS]

df.to_csv(RUTA_PROCESADO, index=False, encoding="utf-8-sig")

print(f"Guardado en: {RUTA_PROCESADO}")
print(f"Filas: {len(df)}  |  Columnas: {len(df.columns)}")

## Resumen del paso de transformación

| Acción | Resultado |
|---|---|
| Columnas eliminadas | 2 (`Unnamed: 0`, `Lista de lotes`) |
| Columnas renombradas | 18 → snake_case |
| Columnas nuevas | 5 (`nif_contratista`, `nombre_contratista`, `anio_contrato`, `es_contrato_menor`, `flag_limite`) |
| Importes convertidos | 4 columnas → `Float64` |
| Fecha parseada | `fecha_formalizacion` → `datetime64` |
| Fichero de salida | `data/processed/contratos_limpios.csv` |